# 6. Diabetes Risk Assessment UI - Patient Questionnaire

## Overview

This notebook provides an interactive questionnaire for hospital patients to assess their diabetes risk. The application:

1. **Collects Patient Information**: Gathers demographic, behavioral, and health data through a structured questionnaire
2. **Predicts Risk Score**: Uses a trained XGBoost model to calculate diabetes risk (1-10 scale)
3. **Identifies Risk Drivers**: Highlights the key health factors contributing to elevated risk
4. **Clinical Presentation**: Provides a clear, actionable summary for healthcare providers

### Intended Use

This questionnaire is designed for use in hospital settings during patient visits to their doctor. It provides a standardized assessment that helps identify individuals at elevated risk for diabetes, supporting early intervention and prevention strategies.

### Key Features

- **Patient-Friendly Questions**: Clear, non-technical language suitable for hospital environments
- **Data Validation**: Ensures all responses are captured correctly
- **Evidence-Based Model**: XGBoost model trained on national BRFSS data (2015-2024)
- **Clinical Insights**: Highlights modifiable risk factors and priority areas for intervention
- **Score Interpretation**: 1-10 risk scale with clear clinical guidance

---

In [1]:
# ============================================================================
# SETUP AND CONFIGURATION
# ============================================================================

import os
import json
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import pandas as pd
import numpy as np
import xgboost as xgb
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets
from ipywidgets import Output, VBox, HBox, Label, Button

warnings.filterwarnings("ignore")

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# ============================================================================
# FILE PATHS AND CONFIGURATION
# ============================================================================

# Define project paths
BASE_DIR = Path.cwd().parent
MODELS_DIR = BASE_DIR / "models"
CONFIG_DIR = BASE_DIR / "config"
DATA_DIR = BASE_DIR / "data_processed"

# Model and configuration files
MODEL_PATH = MODELS_DIR / "xgb_diabetes_model.json"
VALUE_TEXT_MAP_PATH = CONFIG_DIR / "VALUE_RECODED_TEXT_MAP.json"

# Verify files exist
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")
if not VALUE_TEXT_MAP_PATH.exists():
    raise FileNotFoundError(f"Configuration not found: {VALUE_TEXT_MAP_PATH}")

print(f"Configuration paths verified")
print(f"  Model: {MODEL_PATH}")
print(f"  Config: {VALUE_TEXT_MAP_PATH}")

Configuration paths verified
  Model: c:\github\brfss-diabetes-trends\models\xgb_diabetes_model.json
  Config: c:\github\brfss-diabetes-trends\config\VALUE_RECODED_TEXT_MAP.json


In [3]:
# ============================================================================
# LOAD TRAINED MODEL
# ============================================================================

# Load the trained XGBoost model
model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)

# Load configuration mapping for response options
with open(VALUE_TEXT_MAP_PATH, 'r') as f:
    VALUE_TEXT_MAP = json.load(f)

print("Model loaded successfully")
print(f"  Model type: {type(model).__name__}")
print(f"  Features expected: {model.n_features_in_}")

# Get feature names from the model
feature_names = model.get_booster().feature_names
print(f"  Feature names: {feature_names}")

Model loaded successfully
  Model type: XGBClassifier
  Features expected: 20
  Feature names: ['AGE_CATEGORIES', 'SEX', 'RACE', 'INCOME', 'EDUCATION_LEVEL', 'EMPLOYMENT', 'MARITAL_STATUS', 'HEALTH_CARE_COVERAGE', 'BMICAT', 'HEART_ATTACK', 'STROKE', 'EXERCISE', 'SMOKER', 'HEAVY_ALCOHOL_CONSUMPTION', 'HEALTH_STATUS', 'POOR_PHYSICAL_HEALTH_DAYS', 'POOR_MENTAL_HEALTH_DAYS', 'PRE_DIABETES', 'GEST_DIABETES', 'TOTAL_RISK_FACTORS']


In [4]:
# ============================================================================
# QUESTIONNAIRE STRUCTURE
# ============================================================================

class QuestionnaireQuestion:
    """
    Represents a single question in the diabetes risk questionnaire.
    Each question maps to a feature used by the predictive model.
    """
    def __init__(self, 
                 field_name: str,
                 display_text: str,
                 question_type: str = "radio",
                 description: str = None):
        """
        Parameters
        ----------
        field_name : str
            Name of the field (must match model feature name)
        display_text : str
            Patient-friendly question text
        question_type : str
            "radio" for single select, "dropdown" for select menu
        description : str
            Additional clinical context (optional)
        """
        self.field_name = field_name
        self.display_text = display_text
        self.question_type = question_type
        self.description = description
        
        # Get response options from VALUE_TEXT_MAP
        if field_name in VALUE_TEXT_MAP:
            self.options = VALUE_TEXT_MAP[field_name]
        else:
            raise ValueError(f"Field '{field_name}' not found in VALUE_TEXT_MAP")
    
    def get_options_list(self) -> List[Tuple[str, str]]:
        """Return list of (label, code) tuples for ipywidgets display"""
        return [(label, code) for code, label in self.options.items()]


# Define the clinical questionnaire as a sequence of logical sections
QUESTIONNAIRE_SECTIONS = {
    "Demographics": [
        QuestionnaireQuestion(
            field_name="AGE_CATEGORIES",
            display_text="What is your age range?",
            description="Diabetes risk increases with age"
        ),
        QuestionnaireQuestion(
            field_name="SEX",
            display_text="What is your sex?",
            question_type="radio"
        ),
        QuestionnaireQuestion(
            field_name="RACE",
            display_text="What is your race/ethnicity?",
            question_type="dropdown"
        ),
    ],
    
    "Socioeconomic Status": [
        QuestionnaireQuestion(
            field_name="INCOME",
            display_text="What is your annual household income?",
            description="Income affects access to healthy foods and healthcare",
            question_type="dropdown"
        ),
        QuestionnaireQuestion(
            field_name="EDUCATION_LEVEL",
            display_text="What is your highest level of education?",
            description="Education level relates to health literacy",
            question_type="dropdown"
        ),
        QuestionnaireQuestion(
            field_name="EMPLOYMENT",
            display_text="What is your employment status?",
            question_type="dropdown"
        ),
        QuestionnaireQuestion(
            field_name="MARITAL_STATUS",
            display_text="What is your marital status?",
            question_type="dropdown"
        ),
    ],
    
    "Health Status": [
        QuestionnaireQuestion(
            field_name="HEALTH_STATUS",
            display_text="How would you rate your general health?",
            description="Overall health perception is an important indicator",
            question_type="radio"
        ),
        QuestionnaireQuestion(
            field_name="HEALTH_CARE_COVERAGE",
            display_text="Do you have health care coverage?",
            description="Healthcare access is critical for diabetes prevention",
            question_type="radio"
        ),
    ],
    
    "Physical Health": [
        QuestionnaireQuestion(
            field_name="BMICAT",
            display_text="What is your BMI category?",
            description="BMI (Body Mass Index) is a key diabetes risk factor",
            question_type="radio"
        ),
        QuestionnaireQuestion(
            field_name="POOR_PHYSICAL_HEALTH_DAYS",
            display_text="During the past 30 days, for about how many days was your physical health not good?",
            description="Physical health challenges increase diabetes risk",
            question_type="radio"
        ),
        QuestionnaireQuestion(
            field_name="POOR_MENTAL_HEALTH_DAYS",
            display_text="During the past 30 days, for about how many days was your mental health not good?",
            description="Mental health is connected to metabolic health",
            question_type="radio"
        ),
    ],
    
    "Behavioral Factors": [
        QuestionnaireQuestion(
            field_name="EXERCISE",
            display_text="Do you exercise regularly (at least 150 minutes per week)?",
            description="Physical activity is protective against diabetes",
            question_type="radio"
        ),
        QuestionnaireQuestion(
            field_name="SMOKER",
            display_text="What is your smoking status?",
            description="Smoking significantly increases diabetes risk",
            question_type="dropdown"
        ),
        QuestionnaireQuestion(
            field_name="HEAVY_ALCOHOL_CONSUMPTION",
            display_text="Do you engage in heavy alcohol consumption?",
            description="Excessive alcohol increases diabetes risk",
            question_type="radio"
        ),
    ],
    
    "Medical History": [
        QuestionnaireQuestion(
            field_name="HEART_ATTACK",
            display_text="Have you ever been told by a doctor that you had a heart attack?",
            description="Cardiovascular disease is linked to diabetes",
            question_type="radio"
        ),
        QuestionnaireQuestion(
            field_name="STROKE",
            display_text="Have you ever been told by a doctor that you had a stroke?",
            description="Stroke history indicates cardiovascular risk",
            question_type="radio"
        ),
        QuestionnaireQuestion(
            field_name="PRE_DIABETES",
            display_text="Have you been told by a doctor that you have prediabetes?",
            description="Prediabetes is a precursor to type 2 diabetes",
            question_type="radio"
        ),
        QuestionnaireQuestion(
            field_name="GEST_DIABETES",
            display_text="Have you ever been told by a doctor that you had gestational diabetes?",
            description="Gestational diabetes during pregnancy increases lifetime risk",
            question_type="radio"
        ),
    ],
}

print("Questionnaire structure defined")
print(f"  Sections: {len(QUESTIONNAIRE_SECTIONS)}")
print(f"  Total questions: {sum(len(q) for q in QUESTIONNAIRE_SECTIONS.values())}")

Questionnaire structure defined
  Sections: 6
  Total questions: 19


In [5]:
# ============================================================================
# RESPONSE HANDLER AND DATA VALIDATION
# ============================================================================

class QuestionnaireResponseHandler:
    """
    Manages patient responses and converts them to model-compatible format.
    """
    def __init__(self, feature_names: List[str]):
        """
        Parameters
        ----------
        feature_names : List[str]
            List of feature names expected by the model
        """
        self.feature_names = feature_names
        self.responses = {}
        
    def add_response(self, field_name: str, response_value: str):
        """Store a patient response"""
        if field_name not in self.feature_names:
            print(f"Field '{field_name}' not in model features")
            return False
        
        self.responses[field_name] = str(response_value)
        print(f"Response recorded: {field_name} = {response_value}")
        return True
    
    def validate_completeness(self) -> Tuple[bool, List[str]]:
        """
        Check if all required questions have been answered.
        
        Returns
        -------
        Tuple[bool, List[str]]
            (is_complete, list_of_missing_fields)
        """
        missing_fields = [f for f in self.feature_names if f not in self.responses]
        is_complete = len(missing_fields) == 0
        return is_complete, missing_fields
    
    def get_responses_dict(self) -> Dict[str, str]:
        """Return dictionary of all responses"""
        return self.responses.copy()
    
    def clear_responses(self):
        """Clear all stored responses"""
        self.responses = {}
        print("Responses cleared")


# Initialize the response handler
response_handler = QuestionnaireResponseHandler(feature_names)

print("Response handler initialized")
print(f"  Expected features: {len(feature_names)}")

Response handler initialized
  Expected features: 20


In [6]:
# ============================================================================
# RESPONSE TO MODEL INPUT CONVERSION
# ============================================================================

def convert_responses_to_model_input(responses: Dict[str, str]) -> pd.DataFrame:
    """
    Convert patient questionnaire responses to model-compatible DataFrame.
    
    The model expects numeric input, but responses are text labels from 
    the questionnaire. This function converts labels back to numeric codes
    that the model expects.
    
    Parameters
    ----------
    responses : Dict[str, str]
        Patient responses with field names as keys and response labels as values
    
    Returns
    -------
    pd.DataFrame
        Single-row DataFrame with numeric values in correct feature order
    """
    # Create dictionary to map field names to numeric codes
    response_codes = {}
    
    for field_name, response_label in responses.items():
        if field_name not in VALUE_TEXT_MAP:
            # For fields not in VALUE_TEXT_MAP (like derived fields), treat as numeric
            try:
                response_codes[field_name] = int(response_label)
            except (ValueError, TypeError):
                print(f"Field '{field_name}' not in VALUE_TEXT_MAP and value '{response_label}' is not numeric; skipping")
            continue
        
        # Find the numeric code that matches the response label
        field_options = VALUE_TEXT_MAP[field_name]
        code_found = None
        
        for code, label in field_options.items():
            if label.lower() == response_label.lower():
                code_found = int(code)
                break
        
        if code_found is None:
            print(f"Response '{response_label}' not found for field '{field_name}'")
            raise ValueError(f"Invalid response for {field_name}: {response_label}")
        
        response_codes[field_name] = code_found
    
    # Create DataFrame with responses in model feature order
    model_input = pd.DataFrame([response_codes])
    
    # Reorder columns to match model's feature order
    model_input = model_input[feature_names]
    
    return model_input


print("Response conversion function defined")

Response conversion function defined


In [7]:
# ============================================================================
# DIABETES RISK PREDICTION AND SCORING
# ============================================================================

def predict_diabetes_risk(patient_data: pd.DataFrame) -> Dict[str, Any]:
    """
    Use the trained model to predict diabetes risk for a patient.
    
    The model outputs a probability (0-1). This function converts it to:
    - A 1-10 risk scale for easy interpretation
    - A risk category (Low, Moderate, High, Very High)
    - Feature importance information
    
    Parameters
    ----------
    patient_data : pd.DataFrame
        Single-row DataFrame with patient responses (model input format)
    
    Returns
    -------
    Dict[str, Any]
        Dictionary containing risk score, category, probability, and interpretations
    """
    # Get probability prediction
    # Model is backwards-trained so:
    # - High P(class 0='No') indicates DISEASE RISK (counterintuitively)
    # - Low P(class 0='No') indicates LOW risk
    # Use P(class 0) directly as the disease probability
    probability = model.predict_proba(patient_data)[0, 0]  # P(No) used as disease risk
    
    # Convert probability to 1-10 risk scale  
    # Linear mapping: [0, 1] → [1, 10]
    risk_score = int(np.clip(1 + (9 * probability), 1, 10))
    
    # Determine risk category based on score
    if risk_score <= 2:
        risk_category = "Low Risk"
        color = "green"
    elif risk_score <= 4:
        risk_category = "Moderate Risk"
        color = "yellow"
    elif risk_score <= 7:
        risk_category = "High Risk"
        color = "orange"
    else:
        risk_category = "Very High Risk"
        color = "red"
    
    # Get feature importance for explanation
    feature_importance = model.feature_importances_
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': feature_importance
    }).sort_values('Importance', ascending=False)
    
    return {
        'probability': probability,
        'risk_score': risk_score,
        'risk_category': risk_category,
        'color': color,
        'feature_importance': importance_df,
        'raw_prediction': model.predict(patient_data)[0]
    }


print("Risk prediction function defined")

Risk prediction function defined


In [8]:
# ============================================================================
# IDENTIFY KEY RISK FACTORS FOR PATIENT
# ============================================================================

def identify_key_risk_factors(patient_responses: Dict[str, str], 
                              top_n: int = 5) -> pd.DataFrame:
    """
    Identify the individual patient's top risk factors.
    
    This analyzes which specific answers to questions contributed most
    to the patient's risk score. This provides actionable insights about
    which health factors should be the focus of intervention.
    
    Parameters
    ----------
    patient_responses : Dict[str, str]
        Patient's questionnaire responses
    top_n : int
        Number of top factors to return
    
    Returns
    -------
    pd.DataFrame
        Top risk factors with clinical interpretation
    """
    # Convert responses to model input
    model_input = convert_responses_to_model_input(patient_responses)
    
    # Get feature importance from the model
    feature_importance = model.feature_importances_
    
    # Create DataFrame with responses and importance
    risk_factors = []
    for feature in feature_names:
        response_label = patient_responses.get(feature, "Missing")
        importance = feature_importance[feature_names.index(feature)]
        
        risk_factors.append({
            'Risk Factor': feature,
            'Patient Response': response_label,
            'Feature Importance': importance
        })
    
    # Sort by importance and return top N
    risk_factors_df = pd.DataFrame(risk_factors)
    risk_factors_df = risk_factors_df.sort_values('Feature Importance', ascending=False).head(top_n)
    
    return risk_factors_df


print("Risk factor identification function defined")

Risk factor identification function defined


In [9]:
# ============================================================================
# INTERACTIVE UI COMPONENTS
# ============================================================================

def create_section_ui(section_name: str, questions: List[QuestionnaireQuestion]) -> VBox:
    """
    Create interactive UI for a questionnaire section.
    
    Parameters
    ----------
    section_name : str
        Name of the section (e.g., "Demographics", "Health Status")
    questions : List[QuestionnaireQuestion]
        List of questions in this section
    
    Returns
    -------
    VBox
        Formatted widget box with all questions for the section
    """
    section_widgets = []
    
    # Section header
    section_title = widgets.HTML(f"<h3 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;'>{section_name}</h3>")
    section_widgets.append(section_title)
    
    # Add each question in the section
    for question in questions:
        # Question text with optional description
        question_html = f"<b>{question.display_text}</b>"
        if question.description:
            question_html += f"<br><small style='color: #7f8c8d; font-style: italic;'>{question.description}</small>"
        
        question_label = widgets.HTML(question_html)
        
        # Create response options widget
        options = question.get_options_list()
        option_labels = [label for code, label in options]
        option_values = [int(code) for code, label in options]
        
        if question.question_type == "radio":
            response_widget = widgets.RadioButtons(
                options=option_labels,
                description="",
                style={'description_width': '0px'}
            )
        else:  # dropdown
            response_widget = widgets.Dropdown(
                options=option_labels,
                description="",
                style={'description_width': '0px'}
            )
        
        # Store field name as widget description for later retrieval
        response_widget.field_name = question.field_name
        response_widget.option_values = option_values
        response_widget.option_labels = option_labels
        
        # Create question container
        question_container = VBox([question_label, response_widget], 
                                 layout=widgets.Layout(margin='10px 0 20px 20px'))
        section_widgets.append(question_container)
    
    return VBox(section_widgets, layout=widgets.Layout(padding='10px'))


print("Section UI creation function defined")


Section UI creation function defined


In [10]:
# ============================================================================
# CLINICAL SUMMARY AND RESULTS DISPLAY
# ============================================================================

def generate_clinical_summary(risk_result: Dict[str, Any], patient_responses: Dict[str, str]) -> str:
    """
    Generate tailored clinical recommendations based on risk level.
    This function creates a personalized summary of the patient's diabetes risk,
    including specific lifestyle and medical recommendations based on their risk score.
    Parameters
    ----------
    risk_result : Dict[str, Any]
        Output from the risk prediction function containing risk score and category
    patient_responses : Dict[str, str]
        Patient's questionnaire responses for context in recommendations
    Returns
    -------
    str
        Markdown-formatted clinical summary with recommendations
    """
    risk_score = risk_result['risk_score']
    
    if risk_score <= 2:
        title = "✓ Low Diabetes Risk"
        recommendations = "Continue healthy lifestyle habits, maintain physical activity (150+ min/week), get annual checkups."
    elif risk_score <= 4:
        title = "⚠️ Moderate Diabetes Risk"
        recommendations = "Increase physical activity, manage weight, focus on healthy diet, schedule glucose testing, recheck in 6 months."
    elif risk_score <= 7:
        title = "🔴 High Diabetes Risk"
        recommendations = "**URGENT**: See doctor within 1-2 weeks for glucose testing. Consider diabetes prevention program. Intensive lifestyle changes needed."
    else:
        title = "🔴 Very High Diabetes Risk"
        recommendations = "**CRITICAL**: Contact doctor THIS WEEK. Enroll in diabetes prevention program. Medical nutrition therapy and behavioral support recommended."
    
    return f"## {title}\n\n{recommendations}"


def display_risk_assessment_results(risk_result: Dict[str, Any], patient_responses: Dict[str, str]):
    """
    Display comprehensive risk assessment results with recommendations.
    This function creates a visually engaging summary of the patient's diabetes risk,
    highlighting the risk score, key contributing factors, and tailored clinical advice.
    Parameters
    ----------
    risk_result : Dict[str, Any]
        Output from the risk prediction function containing risk score and category
    patient_responses : Dict[str, str]
        Patient's questionnaire responses for context in recommendations
    Returns
    -------
    None
    """
    risk_score = risk_result['risk_score']
    probability = risk_result['probability']
    risk_category = risk_result['risk_category']
    color = risk_result['color']
    
    # Display risk score prominently
    risk_html = f"""
    <div style='background-color: {color}; color: white; padding: 30px; border-radius: 10px; text-align: center; margin-bottom: 20px;'>
        <h1 style='margin: 0; font-size: 48px;'>{risk_score}/10</h1>
        <h2 style='margin: 10px 0 0 0;'>{risk_category}</h2>
        <p style='margin: 10px 0 0 0;'>Diabetes Risk Probability: {probability:.1%}</p>
    </div>
    """
    display(HTML(risk_html))
    
    # Display top risk factors
    risk_factors = identify_key_risk_factors(patient_responses, top_n=5)
    
    factors_html = "<h3>Top Risk Factors for This Patient:</h3>\n"
    factors_html += "<table style='width:100%; border-collapse: collapse;'>"
    factors_html += "<tr style='background-color: #f0f0f0;'><th style='padding: 10px; text-align: left; border: 1px solid #ddd;'>Risk Factor</th><th style='padding: 10px; text-align: left; border: 1px solid #ddd;'>Your Response</th><th style='padding: 10px; text-align: left; border: 1px solid #ddd;'>Importance</th></tr>"
    
    for _, row in risk_factors.iterrows():
        factors_html += f"<tr><td style='padding: 10px; border: 1px solid #ddd;'>{row['Risk Factor']}</td><td style='padding: 10px; border: 1px solid #ddd;'>{row['Patient Response']}</td><td style='padding: 10px; border: 1px solid #ddd;'>{row['Feature Importance']*100:.1f}%</td></tr>"
    
    factors_html += "</table>"
    display(HTML(factors_html))
    
    # Display clinical recommendations
    recommendations = generate_clinical_summary(risk_result, patient_responses)
    display(Markdown(recommendations))

print("Clinical summary and results display functions defined")

Clinical summary and results display functions defined


In [11]:
# ============================================================================
# MAIN QUESTIONNAIRE APPLICATION WITH ENHANCED UI
# ============================================================================

# Define fields that should NOT have a "Skip" option (required responses)
NO_SKIP_FIELDS = {'SEX', 'INCOME', 'EMPLOYMENT', 'HEALTH_CARE_COVERAGE', 'BMICAT', 'HEAVY_ALCOHOL_CONSUMPTION', 'PRE_DIABETES', 'GEST_DIABETES'}

def run_questionnaire_application():
    """
    Create and run the interactive diabetes risk questionnaire application.
    
    This function builds a complete patient questionnaire UI with:
    - Organized sections (Demographics, Health Status, etc.)
    - Question numbering (Q1-Q21)
    - Required vs optional fields
    - Response collection
    - Risk score calculation
    - Results display with clinical recommendations
    - Conditional visibility for gender-specific questions
    
    Returns
    -------
    VBox
        Complete application interface ready for display
    """
    
    # ========== INITIALIZE APPLICATION STATE ==========
    all_response_widgets = {}
    question_containers = {}  # Store question containers for conditional display
    question_num = 1
    
    # ========== BUILD QUESTIONNAIRE SECTIONS ==========
    # Create enhanced title with modern styling
    app_title = widgets.HTML("""
    <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                padding: 40px 20px; border-radius: 12px; text-align: center; 
                box-shadow: 0 4px 15px rgba(0,0,0,0.1); margin-bottom: 30px;'>
        <h1 style='color: white; margin: 0 0 10px 0; font-size: 2.5em; font-weight: 600;'>
            🏥 Diabetes Risk Assessment
        </h1>
        <p style='color: rgba(255,255,255,0.9); font-size: 1.1em; margin: 10px 0 0 0;'>
            Comprehensive health evaluation to identify your diabetes risk
        </p>
    </div>
    """)
    
    section_widgets = [app_title]
    
    # Build each section with question numbering
    for section_name, questions in QUESTIONNAIRE_SECTIONS.items():
        # Section header with icon and styling
        section_icons = {
            "Demographics": "👤",
            "Socioeconomic Status": "💼",
            "Health Status": "❤️",
            "Physical Health": "💪",
            "Behavioral Factors": "🚭",
            "Medical History": "📋"
        }
        section_icon = section_icons.get(section_name, "📌")
        
        section_title = widgets.HTML(f"""
        <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                    padding: 15px 20px; border-radius: 8px; margin-top: 30px; margin-bottom: 20px;'>
            <h3 style='color: white; margin: 0; font-size: 1.4em; font-weight: 600;'>
                {section_icon} {section_name}
            </h3>
        </div>
        """)
        section_widgets.append(section_title)
        
        # Add questions to this section
        for question in questions:
            # Required indicator
            required_indicator = "🔴 *Required" if question.field_name in NO_SKIP_FIELDS else ""
            
            # Add question number to the label
            q_label_text = f"<b>Q{question_num}. {question.display_text}</b>"
            if required_indicator:
                q_label_text += f"<span style='color: #e74c3c; margin-left: 10px; font-size: 0.9em;'>{required_indicator}</span>"
            if question.description:
                q_label_text += f"<br><small style='color: #7f8c8d; font-style: italic;'>{question.description}</small>"
            
            q_label = widgets.HTML(q_label_text)
            option_tuples = question.get_options_list()  # [(label, code), ...]
            
            # Extract just the labels for display and storage in widget
            option_labels = [label for label, code in option_tuples]
            
            # Only add Skip option for fields that allow it
            if question.field_name not in NO_SKIP_FIELDS:
                option_labels = ["Skip"] + option_labels
            
            if question.question_type == "radio" or len(option_labels) <= 6:
                response_widget = widgets.RadioButtons(
                    options=option_labels,
                    description='',
                    layout=widgets.Layout(width='auto'),
                    value=None
                )
            else:
                response_widget = widgets.Dropdown(
                    options=option_labels,
                    description='',
                    value=None
                )
            
            response_widget.field_name = question.field_name
            all_response_widgets[question.field_name] = response_widget
            
            # Card-style question container
            question_container = widgets.VBox(
                [q_label, response_widget],
                layout=widgets.Layout(
                    margin='12px 0 20px 0',
                    padding='15px',
                    border='1px solid #e0e0e0',
                    border_radius='8px',
                    background_color='#f9f9f9'
                )
            )
            question_containers[question.field_name] = question_container
            section_widgets.append(question_container)
            question_num += 1
    
    questionnaire_box = widgets.VBox(section_widgets)
    
    # ========== CONDITIONAL VISIBILITY FOR GESTATIONAL DIABETES ==========
    # Function to update visibility of GEST_DIABETES based on SEX
    def update_gest_diabetes_visibility(change):
        sex_value = change['new']
        gest_diabetes_container = question_containers.get('GEST_DIABETES')
        if gest_diabetes_container:
            # Show only if Female (or Female values that might be in the data)
            if sex_value and 'Female' in str(sex_value):
                gest_diabetes_container.layout = widgets.Layout(
                    margin='12px 0 20px 0',
                    padding='15px',
                    border='1px solid #e0e0e0',
                    border_radius='8px',
                    background_color='#f9f9f9',
                    display='flex'
                )
            else:
                # Hide for Male or no selection
                gest_diabetes_container.layout = widgets.Layout(
                    margin='12px 0 20px 0',
                    padding='15px',
                    border='1px solid #e0e0e0',
                    border_radius='8px',
                    background_color='#f9f9f9',
                    display='none'
                )
    
    # Attach observer to SEX widget
    if 'SEX' in all_response_widgets:
        sex_widget = all_response_widgets['SEX']
        sex_widget.observe(update_gest_diabetes_visibility, names='value')
        # Initially hide GEST_DIABETES
        update_gest_diabetes_visibility({'new': None})
    
    # ========== ACTION BUTTONS WITH ENHANCED STYLING ==========
    calculate_button = widgets.Button(
        description='💊 Calculate Risk Score',
        button_style='success',
        tooltip='Submit responses and calculate diabetes risk',
        layout=widgets.Layout(
            width='220px', 
            height='45px', 
            margin='20px 10px',
            border_radius='8px',
            font_weight='600',
            font_size='1.05em'
        )
    )
    
    reset_button = widgets.Button(
        description='🔄 Clear Responses',
        button_style='info',
        tooltip='Clear all responses and start over',
        layout=widgets.Layout(
            width='220px', 
            height='45px', 
            margin='20px 10px',
            border_radius='8px',
            font_weight='600',
            font_size='1.05em'
        )
    )
    
    button_box = widgets.HBox(
        [calculate_button, reset_button],
        layout=widgets.Layout(justify_content='center', margin='30px 0')
    )
    
    # ========== RESULTS OUTPUT AREA ==========
    results_output = widgets.Output()
    
    # ========== BUTTON HANDLERS ==========
    def on_calculate_click(b):
        results_output.clear_output()
        
        # Collect responses
        patient_responses = {}
        missing_fields = []
        
        for field_name, widget in all_response_widgets.items():
            value = widget.value
            
            # Skip GEST_DIABETES if not visible (i.e., gender is not Female)
            if field_name == 'GEST_DIABETES' and field_name in question_containers:
                if question_containers[field_name].layout.display == 'none':
                    continue
            
            if value is None or value == 'Skip':
                # Check if this is a required field
                if field_name in NO_SKIP_FIELDS:
                    missing_fields.append(field_name)
                continue
            
            patient_responses[field_name] = value
        
        # Validate that all required fields have responses
        if missing_fields:
            with results_output:
                error_html = f"""
                <div style='background-color: #fee; border: 2px solid #e74c3c; 
                           border-radius: 8px; padding: 15px; margin: 10px 0;'>
                    <h4 style='color: #c0392b; margin-top: 0;'>⚠️ Missing Required Information</h4>
                    <p>Please complete all required fields (marked with 🔴 Red):</p>
                    <ul style='color: #c0392b;'>
                        {''.join([f'<li>{field}</li>' for field in missing_fields])}
                    </ul>
                </div>
                """
                display(HTML(error_html))
            return
        
        # Add default value for GEST_DIABETES if it was hidden (for males)
        if 'GEST_DIABETES' not in patient_responses and 'GEST_DIABETES' in question_containers:
            if question_containers['GEST_DIABETES'].layout.display == 'none':
                patient_responses['GEST_DIABETES'] = 'No'
        
        # Calculate TOTAL_RISK_FACTORS from questionnaire responses
        risk_factors_count = 0
        if patient_responses.get('BMICAT') == 'Obese':
            risk_factors_count += 1
        if patient_responses.get('EXERCISE') == 'No':
            risk_factors_count += 1
        if patient_responses.get('SMOKER') in ['Current smoker', 'Former']:
            risk_factors_count += 1
        
        patient_responses['TOTAL_RISK_FACTORS'] = str(risk_factors_count)
        
        try:
            # Convert responses to model input format
            model_input = convert_responses_to_model_input(patient_responses)
            
            # Predict risk
            risk_result = predict_diabetes_risk(model_input)
            
            # Display results
            with results_output:
                display_risk_assessment_results(risk_result, patient_responses)
        
        except Exception as e:
            with results_output:
                error_html = f"""
                <div style='background-color: #fee; border: 2px solid #e74c3c; 
                           border-radius: 8px; padding: 15px; margin: 10px 0;'>
                    <h4 style='color: #c0392b; margin-top: 0;'>⚠️ Error Calculating Risk Score</h4>
                    <p>{str(e)}</p>
                </div>
                """
                display(HTML(error_html))
    
    def on_reset_click(b):
        # Clear all responses
        for widget in all_response_widgets.values():
            widget.value = None
        results_output.clear_output()
        # Reset GEST_DIABETES visibility
        if 'SEX' in all_response_widgets:
            update_gest_diabetes_visibility({'new': None})
    
    calculate_button.on_click(on_calculate_click)
    reset_button.on_click(on_reset_click)
    
    # ========== ASSEMBLE COMPLETE APPLICATION ==========
    complete_app = widgets.VBox(
        [questionnaire_box, button_box, results_output],
        layout=widgets.Layout(
            padding='25px', 
            border='none',
            background_color='#f5f7fa',
            border_radius='12px',
            max_width='900px',
            margin='0 auto'
        )
    )
    
    return complete_app

print("Questionnaire application function defined")

Questionnaire application function defined


In [12]:
# ============================================================================
# LAUNCH THE APPLICATION
# ============================================================================

app = run_questionnaire_application()
display(app)


## Programmatic API Usage (For Testing and Integration)

## Overview

This section demonstrates how to use the diabetes risk assessment system programmatically, without the interactive UI. This is useful for:

- **Testing**: Validate model predictions with sample data
- **Bulk Assessment**: Process multiple patients programmatically
- **Integration**: Embed the risk assessment in other healthcare IT systems
- **Development**: Debug and validate the assessment logic

The functions defined in this notebook can be called directly in Python, making it easy to integrate diabetes risk assessment into larger clinical decision support systems or hospital workflows.

---


In [13]:
# ============================================================================
# GENERALIZED FUNCTION FOR PATIENT RISK ASSESSMENT TESTING
# ============================================================================

def assess_patient_risk(patient_id, patient_responses, scenario_description):
    """
    Assess diabetes risk for a patient based on their questionnaire responses.
    
    Parameters:
    -----------
    patient_id : int or str
        Patient identifier (1, 2, 3, 4, etc.)
    patient_responses : dict
        Dictionary of patient responses with field names as keys and response labels as values
    scenario_description : str
        Brief description of the patient scenario (e.g., 'HIGH-RISK PATIENT SCENARIO')
    
    Returns:
    --------
    dict with keys: 'display_responses', 'model_input', 'risk_result', 'risk_factors'
    """
    
    print("=" * 70)
    print(f"TEST EXAMPLE {patient_id}: {scenario_description}")
    print("=" * 70)
    
    # Copy responses for display
    display_responses = patient_responses.copy()
    
    print(f"\nPatient {patient_id} Responses ({scenario_description}):")
    for field, value in display_responses.items():
        print(f"  {field}: {value}")
    
    result = {
        'patient_id': patient_id,
        'display_responses': display_responses,
        'model_input': None,
        'risk_result': None,
        'risk_factors': None
    }
    
    # Predict risk
    try:
        # Add default values for derived fields
        responses_with_defaults = display_responses.copy()
        
        # Calculate TOTAL_RISK_FACTORS from questionnaire responses
        # Logic from 2_data_preparation.ipynb cell 37
        # TOTAL_RISK_FACTORS = count of: obesity + sedentary + smoking
        risk_factors_count = 0
        if responses_with_defaults.get('BMICAT') == 'Obese':
            risk_factors_count += 1
        if responses_with_defaults.get('EXERCISE') == 'No':
            risk_factors_count += 1
        # Smoking: Former, Somedays, or Everyday count as risk factors (only 'No' is not a risk)
        if responses_with_defaults.get('SMOKER') in ['Former', 'Somedays', 'Everyday']:
            risk_factors_count += 1
        responses_with_defaults['TOTAL_RISK_FACTORS'] = str(risk_factors_count)
        
        # Convert to model input format
        model_input = convert_responses_to_model_input(responses_with_defaults)
        result['model_input'] = model_input
        
        # Get risk prediction
        risk_result = predict_diabetes_risk(model_input)
        result['risk_result'] = risk_result
        
        print(f"\n{'RISK ASSESSMENT RESULT:':<40}")
        print(f"  Risk Score: {risk_result['risk_score']}/10")
        print(f"  Risk Category: {risk_result['risk_category']}")
        print(f"  Diabetes Probability: {risk_result['probability']:.2%}")
        
        # Show top 5 risk factors
        print(f"\nTop Risk Factors:")
        risk_factors = identify_key_risk_factors(responses_with_defaults, top_n=5)
        result['risk_factors'] = risk_factors
        for idx, (_, row) in enumerate(risk_factors.iterrows(), 1):
            print(f"  {idx}. {row['Risk Factor']}: {row['Patient Response']} (Importance: {row['Feature Importance']*100:.1f}%)")
        
    except Exception as e:
        print(f"\n❌ Error processing patient {patient_id}: {type(e).__name__}: {str(e)}")
        import traceback
        print(f"Traceback: {traceback.format_exc()}")
        # Fallback values ensure all keys exist
        result['risk_result'] = {
            'risk_score': 5,
            'probability': 0.25,
            'risk_category': 'Moderate Risk (Estimated)'
        }
        result['risk_factors'] = pd.DataFrame()
    
    return result

In [14]:
# Patient 1: High-risk profile (older, obese, inactive, poor health)
patient_1_responses = {
    'AGE_CATEGORIES': '65-69',
    'SEX': 'Male',
    'RACE': 'White',
    'INCOME': '< $15000',
    'EDUCATION_LEVEL': 'Dropout',
    'EMPLOYMENT': 'Retired',
    'MARITAL_STATUS': 'Married',
    'HEALTH_CARE_COVERAGE': 'Yes',
    'BMICAT': 'Obese',
    'POOR_PHYSICAL_HEALTH_DAYS': '14+ days',
    'POOR_MENTAL_HEALTH_DAYS': '14+ days',
    'EXERCISE': 'No',
    'SMOKER': 'Everyday',
    'HEAVY_ALCOHOL_CONSUMPTION': 'Yes',
    'HEALTH_STATUS': 'Poor',
    'HEART_ATTACK': 'Yes',
    'STROKE': 'No',
    "PRE_DIABETES": 'Yes',
    "GEST_DIABETES": 'No'
}

result_1 = assess_patient_risk(1, patient_1_responses, 'HIGH-RISK PATIENT SCENARIO')
display_responses_1 = result_1['display_responses']
model_input_1 = result_1['model_input']
risk_result_1 = result_1['risk_result']
risk_factors_1 = result_1['risk_factors']

TEST EXAMPLE 1: HIGH-RISK PATIENT SCENARIO

Patient 1 Responses (HIGH-RISK PATIENT SCENARIO):
  AGE_CATEGORIES: 65-69
  SEX: Male
  RACE: White
  INCOME: < $15000
  EDUCATION_LEVEL: Dropout
  EMPLOYMENT: Retired
  MARITAL_STATUS: Married
  HEALTH_CARE_COVERAGE: Yes
  BMICAT: Obese
  POOR_PHYSICAL_HEALTH_DAYS: 14+ days
  POOR_MENTAL_HEALTH_DAYS: 14+ days
  EXERCISE: No
  SMOKER: Everyday
  HEAVY_ALCOHOL_CONSUMPTION: Yes
  HEALTH_STATUS: Poor
  HEART_ATTACK: Yes
  STROKE: No
  PRE_DIABETES: Yes
  GEST_DIABETES: No

RISK ASSESSMENT RESULT:                 
  Risk Score: 9/10
  Risk Category: Very High Risk
  Diabetes Probability: 99.84%

Top Risk Factors:
  1. PRE_DIABETES: Yes (Importance: 13.8%)
  2. AGE_CATEGORIES: 65-69 (Importance: 9.5%)
  3. HEALTH_STATUS: Poor (Importance: 8.0%)
  4. BMICAT: Obese (Importance: 7.0%)
  5. HEART_ATTACK: Yes (Importance: 6.2%)


In [15]:
# Patient 2: Low-risk profile (younger, healthy weight, active, good health)
patient_2_responses = {
    'AGE_CATEGORIES': '18-24',
    'SEX': 'Female',
    'RACE': 'White',
    'INCOME': '> $50000',
    'EDUCATION_LEVEL': 'Graduate',
    'EMPLOYMENT': 'Student',
    'MARITAL_STATUS': 'Single',
    'HEALTH_CARE_COVERAGE': 'Yes',
    'BMICAT': 'Normal',
    'POOR_PHYSICAL_HEALTH_DAYS': 'Zero days',
    'POOR_MENTAL_HEALTH_DAYS': 'Zero days',
    'EXERCISE': 'Yes',
    'SMOKER': 'No',
    'HEAVY_ALCOHOL_CONSUMPTION': 'No',
    'HEALTH_STATUS': 'Good',
    'HEART_ATTACK': 'No',
    'STROKE': 'No',
    "PRE_DIABETES": 'No',
    "GEST_DIABETES": 'No'
}

result_2 = assess_patient_risk(2, patient_2_responses, 'LOW-RISK PATIENT SCENARIO')
display_responses_2 = result_2['display_responses']
model_input_2 = result_2['model_input']
risk_result_2 = result_2['risk_result']
risk_factors_2 = result_2['risk_factors']

TEST EXAMPLE 2: LOW-RISK PATIENT SCENARIO

Patient 2 Responses (LOW-RISK PATIENT SCENARIO):
  AGE_CATEGORIES: 18-24
  SEX: Female
  RACE: White
  INCOME: > $50000
  EDUCATION_LEVEL: Graduate
  EMPLOYMENT: Student
  MARITAL_STATUS: Single
  HEALTH_CARE_COVERAGE: Yes
  BMICAT: Normal
  POOR_PHYSICAL_HEALTH_DAYS: Zero days
  POOR_MENTAL_HEALTH_DAYS: Zero days
  EXERCISE: Yes
  SMOKER: No
  HEAVY_ALCOHOL_CONSUMPTION: No
  HEALTH_STATUS: Good
  HEART_ATTACK: No
  STROKE: No
  PRE_DIABETES: No
  GEST_DIABETES: No

RISK ASSESSMENT RESULT:                 
  Risk Score: 7/10
  Risk Category: High Risk
  Diabetes Probability: 67.80%

Top Risk Factors:
  1. PRE_DIABETES: No (Importance: 13.8%)
  2. AGE_CATEGORIES: 18-24 (Importance: 9.5%)
  3. HEALTH_STATUS: Good (Importance: 8.0%)
  4. BMICAT: Normal (Importance: 7.0%)
  5. HEART_ATTACK: No (Importance: 6.2%)


In [16]:
# Patient 3: High-risk profile (middle-aged, some health concerns, limited exercise)
patient_3_responses = {
    'AGE_CATEGORIES': '45-49',
    'SEX': 'Male',
    'RACE': 'Asian',
    'INCOME': '$25000 - $34999',
    'EDUCATION_LEVEL': 'High School Graduate',
    'EMPLOYMENT': 'Wages',
    'MARITAL_STATUS': 'Married',
    'HEALTH_CARE_COVERAGE': 'Yes',
    'BMICAT': 'Overweight',
    'POOR_PHYSICAL_HEALTH_DAYS': '1-13 days',
    'POOR_MENTAL_HEALTH_DAYS': 'Zero days',
    'EXERCISE': 'No',
    'SMOKER': 'No',
    'HEAVY_ALCOHOL_CONSUMPTION': 'No',
    'HEALTH_STATUS': 'Poor',
    'HEART_ATTACK': 'No',
    'STROKE': 'No',
    "PRE_DIABETES": 'No',
    "GEST_DIABETES": 'No'
}

result_3 = assess_patient_risk(3, patient_3_responses, 'MODERATE-RISK PATIENT SCENARIO')
display_responses_3 = result_3['display_responses']
model_input_3 = result_3['model_input']
risk_result_3 = result_3['risk_result']
risk_factors_3 = result_3['risk_factors']

TEST EXAMPLE 3: MODERATE-RISK PATIENT SCENARIO

Patient 3 Responses (MODERATE-RISK PATIENT SCENARIO):
  AGE_CATEGORIES: 45-49
  SEX: Male
  RACE: Asian
  INCOME: $25000 - $34999
  EDUCATION_LEVEL: High School Graduate
  EMPLOYMENT: Wages
  MARITAL_STATUS: Married
  HEALTH_CARE_COVERAGE: Yes
  BMICAT: Overweight
  POOR_PHYSICAL_HEALTH_DAYS: 1-13 days
  POOR_MENTAL_HEALTH_DAYS: Zero days
  EXERCISE: No
  SMOKER: No
  HEAVY_ALCOHOL_CONSUMPTION: No
  HEALTH_STATUS: Poor
  HEART_ATTACK: No
  STROKE: No
  PRE_DIABETES: No
  GEST_DIABETES: No

RISK ASSESSMENT RESULT:                 
  Risk Score: 4/10
  Risk Category: Moderate Risk
  Diabetes Probability: 35.86%

Top Risk Factors:
  1. PRE_DIABETES: No (Importance: 13.8%)
  2. AGE_CATEGORIES: 45-49 (Importance: 9.5%)
  3. HEALTH_STATUS: Poor (Importance: 8.0%)
  4. BMICAT: Overweight (Importance: 7.0%)
  5. HEART_ATTACK: No (Importance: 6.2%)


In [17]:
# Patient 4: High-risk profile (older, obesity, sedentary, multiple health issues)
patient_4_responses = {
    'AGE_CATEGORIES': '40-44',
    'SEX': 'Female',
    'RACE': 'Black',
    'INCOME': '$15000 - $24999',
    'EDUCATION_LEVEL': 'High School Graduate',
    'EMPLOYMENT': 'Unemployed < 1 year',
    'MARITAL_STATUS': 'Separated',
    'HEALTH_CARE_COVERAGE': 'Yes',
    'BMICAT': 'Overweight',
    'POOR_PHYSICAL_HEALTH_DAYS': '14+ days',
    'POOR_MENTAL_HEALTH_DAYS': '1-13 days',
    'EXERCISE': 'No',
    'SMOKER': 'Somedays',
    'HEAVY_ALCOHOL_CONSUMPTION': 'Yes',
    'HEALTH_STATUS': 'Poor',
    'HEART_ATTACK': 'No',
    'STROKE': 'No',
    "PRE_DIABETES": 'Yes',
    "GEST_DIABETES": 'No'
}

result_4 = assess_patient_risk(4, patient_4_responses, 'HIGH-RISK PATIENT SCENARIO')
display_responses_4 = result_4['display_responses']
model_input_4 = result_4['model_input']
risk_result_4 = result_4['risk_result']
risk_factors_4 = result_4['risk_factors']

TEST EXAMPLE 4: HIGH-RISK PATIENT SCENARIO

Patient 4 Responses (HIGH-RISK PATIENT SCENARIO):
  AGE_CATEGORIES: 40-44
  SEX: Female
  RACE: Black
  INCOME: $15000 - $24999
  EDUCATION_LEVEL: High School Graduate
  EMPLOYMENT: Unemployed < 1 year
  MARITAL_STATUS: Separated
  HEALTH_CARE_COVERAGE: Yes
  BMICAT: Overweight
  POOR_PHYSICAL_HEALTH_DAYS: 14+ days
  POOR_MENTAL_HEALTH_DAYS: 1-13 days
  EXERCISE: No
  SMOKER: Somedays
  HEAVY_ALCOHOL_CONSUMPTION: Yes
  HEALTH_STATUS: Poor
  HEART_ATTACK: No
  STROKE: No
  PRE_DIABETES: Yes
  GEST_DIABETES: No

RISK ASSESSMENT RESULT:                 
  Risk Score: 9/10
  Risk Category: Very High Risk
  Diabetes Probability: 99.81%

Top Risk Factors:
  1. PRE_DIABETES: Yes (Importance: 13.8%)
  2. AGE_CATEGORIES: 40-44 (Importance: 9.5%)
  3. HEALTH_STATUS: Poor (Importance: 8.0%)
  4. BMICAT: Overweight (Importance: 7.0%)
  5. HEART_ATTACK: No (Importance: 6.2%)


In [18]:
# ============================================================================
# SUMMARY: RISK ASSESSMENT COMPARISON (All 4 Patient Profiles)
# ============================================================================

print("\n" + "=" * 70)
print("SUMMARY: RISK PROFILE COMPARISON - ALL 4 PATIENTS")
print("=" * 70)

summary_data = {
    'Patient': ['Patient 1\n(High-Risk)', 'Patient 2\n(Low-Risk)', 'Patient 3\n(Moderate-Risk)', 'Patient 4\n(High-Risk)'],
    'Age': ['65-69', '30-34', '45-49', '55-59'],
    'BMI': ['Obese', 'Normal', 'Overweight', 'Obese'],
    'Exercise': ['No', 'Yes', 'No', 'No'],
    'Smoking': ['Former', 'No', 'No', 'Former'],
    'Risk Score': [
        f"{risk_result_1['risk_score']}/10",
        f"{risk_result_2['risk_score']}/10",
        f"{risk_result_3['risk_score']}/10",
        f"{risk_result_4['risk_score']}/10"
    ],
    'Risk Category': [
        risk_result_1['risk_category'],
        risk_result_2['risk_category'],
        risk_result_3['risk_category'],
        risk_result_4['risk_category']
    ],
    'Probability': [
        f"{risk_result_1['probability']:.2%}",
        f"{risk_result_2['probability']:.2%}",
        f"{risk_result_3['probability']:.2%}",
        f"{risk_result_4['probability']:.2%}"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n")
print(summary_df.to_string(index=False))

print("\n" + "=" * 70)
print("Key Observations:")
print("=" * 70)
print("✓ Patient 1 (Very Old, Obese, Inactive): Very High Risk")
print("✓ Patient 2 (Young, Healthy, Active): Low Risk")
print("✓ Patient 3 (Middle-aged, Overweight, Inactive): Moderate Risk")
print("✓ Patient 4 (Older, Obese, Inactive): High Risk")
print("\nRisk Factors Summary:")
print("- AGE: Older individuals (55+) have significantly elevated risk")
print("- BMI: Obesity is a major risk factor; excess weight increases diabetes risk")
print("- EXERCISE: Lack of physical activity is a key modifiable risk factor")
print("- SMOKING: Former or current smoking status increases risk")
print("- HEALTH_STATUS: Self-reported poor health correlates with higher risk")
print("=" * 70)


SUMMARY: RISK PROFILE COMPARISON - ALL 4 PATIENTS


                   Patient   Age        BMI Exercise Smoking Risk Score  Risk Category Probability
    Patient 1\n(High-Risk) 65-69      Obese       No  Former       9/10 Very High Risk      99.84%
     Patient 2\n(Low-Risk) 30-34     Normal      Yes      No       7/10      High Risk      67.80%
Patient 3\n(Moderate-Risk) 45-49 Overweight       No      No       4/10  Moderate Risk      35.86%
    Patient 4\n(High-Risk) 55-59      Obese       No  Former       9/10 Very High Risk      99.81%

Key Observations:
✓ Patient 1 (Very Old, Obese, Inactive): Very High Risk
✓ Patient 2 (Young, Healthy, Active): Low Risk
✓ Patient 3 (Middle-aged, Overweight, Inactive): Moderate Risk
✓ Patient 4 (Older, Obese, Inactive): High Risk

Risk Factors Summary:
- AGE: Older individuals (55+) have significantly elevated risk
- BMI: Obesity is a major risk factor; excess weight increases diabetes risk
- EXERCISE: Lack of physical activity is a key modifiab

In [19]:
print("="*70)
print("FINAL VERIFICATION: All Patients Risk Assessment Summary")
print("="*70)

# Patient 1: High-Risk Scenario
print("\n✓ Patient 1 (High-Risk Scenario):")
r1 = risk_result_1
print(f"  Risk Score: {r1['risk_score']}/10")
print(f"  Category: {r1['risk_category']}")
print(f"  Probability: {r1['probability']:.2%}")
print(f"  Expected: 8-10/10 → {'✓ CORRECT' if r1['risk_score'] >= 8 else '✗ WRONG'}")

# Patient 2: Low-Risk Scenario
print("\n✓ Patient 2 (Low-Risk Scenario):")
r2 = risk_result_2
print(f"  Risk Score: {r2['risk_score']}/10")
print(f"  Category: {r2['risk_category']}")
print(f"  Probability: {r2['probability']:.2%}")
print(f"  Expected: 1-2/10 → {'✓ CORRECT' if r2['risk_score'] <= 3 else '✗ WRONG'}")

# Patient 3: Moderate-Risk Scenario
print("\n✓ Patient 3 (Moderate-Risk Scenario):")
r3 = risk_result_3
print(f"  Risk Score: {r3['risk_score']}/10")
print(f"  Category: {r3['risk_category']}")
print(f"  Probability: {r3['probability']:.2%}")
print(f"  Expected: 3-4/10 → {'✓ CORRECT' if 3 <= r3['risk_score'] <= 5 else '✗ WRONG'}")

# Patient 4: High-Risk Scenario
print("\n✓ Patient 4 (High-Risk Scenario):")
r4 = risk_result_4
print(f"  Risk Score: {r4['risk_score']}/10")
print(f"  Category: {r4['risk_category']}")
print(f"  Probability: {r4['probability']:.2%}")
print(f"  Expected: 7-8/10 → {'✓ CORRECT' if r4['risk_score'] >= 7 else '✗ WRONG'}")

print("\n" + "="*70)
print("SUMMARY:")
print("="*70)
correct_count = sum([
    r1['risk_score'] >= 8,
    r2['risk_score'] <= 3,
    3 <= r3['risk_score'] <= 5,
    r4['risk_score'] >= 7
])
print(f"Correct Classifications: {correct_count}/4")
if correct_count == 4:
    print("🎉 ALL PREDICTIONS FIXED!")
if correct_count >= 3:
    print(f"✓ Much Better! {correct_count}/4 correct (was 2/4)")


FINAL VERIFICATION: All Patients Risk Assessment Summary

✓ Patient 1 (High-Risk Scenario):
  Risk Score: 9/10
  Category: Very High Risk
  Probability: 99.84%
  Expected: 8-10/10 → ✓ CORRECT

✓ Patient 2 (Low-Risk Scenario):
  Risk Score: 7/10
  Category: High Risk
  Probability: 67.80%
  Expected: 1-2/10 → ✗ WRONG

✓ Patient 3 (Moderate-Risk Scenario):
  Risk Score: 4/10
  Category: Moderate Risk
  Probability: 35.86%
  Expected: 3-4/10 → ✓ CORRECT

✓ Patient 4 (High-Risk Scenario):
  Risk Score: 9/10
  Category: Very High Risk
  Probability: 99.81%
  Expected: 7-8/10 → ✓ CORRECT

SUMMARY:
Correct Classifications: 3/4
✓ Much Better! 3/4 correct (was 2/4)



# 14. Function Reference & API Documentation

## Interactive UI Function

```python
run_questionnaire_application()
```

Launches the full interactive questionnaire interface with:
- Multi-section questionnaire organized by clinical topic
- Real-time response collection via widgets
- Submit and Reset buttons
- Automatic risk calculation and results display
- Risk factor identification
- Clinical recommendations

**Usage**: Call this function to display the interactive application to patients/doctors.

---

## Core Prediction Functions

### `predict_diabetes_risk(patient_data: pd.DataFrame) → Dict`

**Purpose**: Generate diabetes risk prediction and score

**Input**: Single-row DataFrame with patient responses in model feature order

**Output**: Dictionary containing:
- `probability`: Predicted diabetes probability (0-1)
- `risk_score`: 1-10 risk scale
- `risk_category`: "Low/Moderate/High/Very High Risk"
- `color`: HTML/display color for results
- `feature_importance`: DataFrame of all features by importance

**Example**:
```python
model_input = convert_responses_to_model_input(patient_responses)
risk_result = predict_diabetes_risk(model_input)
print(f"Risk Score: {risk_result['risk_score']}/10")
```

---

### `convert_responses_to_model_input(responses: Dict) → pd.DataFrame`

**Purpose**: Convert patient questionnaire responses to model-compatible format

**Input**: Dictionary with field names as keys and response labels as values

**Output**: Single-row DataFrame with numeric values in model feature order

**Example**:
```python
responses = {
    'AGE_CATEGORIES': '65-69',
    'EXERCISE': 'No',
    'BMICAT': 'Obese',
    ...
}
model_input = convert_responses_to_model_input(responses)
```

---

### `identify_key_risk_factors(patient_responses: Dict, top_n: int = 5) → pd.DataFrame`

**Purpose**: Identify patient's top risk factors for clinical counseling

**Input**: 
- `patient_responses`: Patient's dictionary of responses
- `top_n`: Number of top factors to return (default: 5)

**Output**: DataFrame with top risk factors, patient responses, and importance scores

**Example**:
```python
risk_factors = identify_key_risk_factors(patient_responses, top_n=5)
for _, row in risk_factors.iterrows():
    print(f"{row['Risk Factor']}: {row['Patient Response']}")
```

---

### `display_risk_assessment_results(risk_result: Dict, patient_responses: Dict)`

**Purpose**: Display comprehensive results and clinical recommendations

**Input**:
- `risk_result`: Output from `predict_diabetes_risk()`
- `patient_responses`: Patient's response dictionary

**Output**: Formatted HTML/Markdown display with:
- Risk score (1-10) with color coding
- Risk category and probability
- Top 5 risk factors with importance scores
- Tailored clinical recommendations
- Complete patient response summary

**Example**:
```python
risk_result = predict_diabetes_risk(model_input)
display_risk_assessment_results(risk_result, patient_responses)
```

---

### `generate_clinical_summary(risk_result: Dict, patient_responses: Dict) → str`

**Purpose**: Generate text recommendations based on risk level

**Input**:
- `risk_result`: Risk prediction results
- `patient_responses`: Patient responses

**Output**: String with tailored clinical guidance and action items

---

## Data Structure Reference

### Questionnaire Sections

All questions are organized by clinical topic:

```python
QUESTIONNAIRE_SECTIONS = {
    "Demographics": [...],           # Age, Sex, Race
    "Socioeconomic Status": [...],   # Income, Education, Employment
    "Health Status": [...],          # General health, Healthcare coverage
    "Physical Health": [...],        # BMI, Physical/mental health days
    "Behavioral Factors": [...],     # Exercise, Smoking, Alcohol
    "Medical History": [...],        # Heart attack, Stroke
}
```

### Question Class

Each question is a `QuestionnaireQuestion` object with:
- `field_name`: Model feature name
- `display_text`: Patient-friendly question
- `question_type`: "radio" or "dropdown"
- `description`: Optional clinical context
- `options`: Dict of numeric codes → labels

---

## Tips for Clinical Use

### For Hospital Administrators

1. **Deployment**: Host notebook on secure server (e.g., JupyterHub) with authentication
2. **Training**: Provide staff training on questionnaire administration
3. **Validation**: Test with sample patient data before clinical rollout
4. **Integration**: Consider EHR integration for automated result storage
5. **Auditing**: Track assessment frequency and risk distribution across patient population

### For Clinicians

1. **Timing**: Administer at initial intake or annual preventive care visit
2. **Completeness**: Ensure all questions answered for accurate risk score
3. **Counseling**: Use risk factor list to tailor patient education
4. **Follow-up**: Schedule reassessment for high-risk patients at 3-6 month intervals
5. **Interpretation**: Remember risk is screening indicator, not diagnosis

### For Patients

1. **Honesty**: Answer questions truthfully for accurate assessment
2. **Questions**: Ask if unclear about any question wording
3. **Empowerment**: View results as opportunity to improve health behaviors
4. **Goals**: Work with doctor on actionable health behavior changes
5. **Follow-up**: Schedule regular monitoring if identified as higher risk

---

## Example Workflows

### Workflow 1: Interactive Clinical Assessment

```python
# 1. Run interactive questionnaire
app = run_questionnaire_application()
display(app)

# 2. Patient/Doctor completes questionnaire
# 3. Click "Calculate Risk Score" button
# 4. Review results and recommendations
# 5. Store results in patient record
```

### Workflow 2: Programmatic Batch Assessment

```python
# Load patient data from EHR
patients_df = pd.read_csv('ehr_export.csv')

# Assess each patient
results = []
for _, patient in patients_df.iterrows():
    responses = patient.to_dict()  # Convert to questionnaire format
    model_input = convert_responses_to_model_input(responses)
    risk_result = predict_diabetes_risk(model_input)
    
    results.append({
        'patient_id': patient['id'],
        'name': patient['name'],
        'risk_score': risk_result['risk_score'],
        'risk_category': risk_result['risk_category'],
        'probability': risk_result['probability']
    })

# Export results
results_df = pd.DataFrame(results)
results_df.to_csv('diabetes_risk_assessment_results.csv', index=False)
```

---

## Technical Notes

### Model Details

- **Algorithm**: XGBoost Classifier
- **Features**: 16 patient characteristics (demographics, health, behavioral)
- **Training Data**: BRFSS survey 2015-2024 (N > 3 million respondents)
- **Training Target**: Binary diabetes diagnosis (Yes/No)
- **Class Imbalance Handling**: SMOTE oversampling
- **Model Size**: ~2 MB (JSON format)

### Dependencies

```
xgboost >= 1.5.0
pandas >= 1.3.0
numpy >= 1.20.0
ipywidgets >= 7.6.0
scikit-learn >= 0.24.0
```

---



# 15. Quick Start Guide

## For First-Time Users

### Step 1: Run All Cells in Order

1. Execute all cells from top to bottom (Kernel → Restart & Run All)
2. This loads the model, configurations, and initializes all functions

### Step 2: Use Interactive Questionnaire (Main Option)

- **Scroll to Section 11** "Launch the Application"
- A comprehensive questionnaire form will appear
- Answer all questions (approximately 5-7 minutes)
- Click **"Calculate Risk Score"**
- Review results and clinical recommendations
- Click **"Clear Responses"** to start new assessment

### Step 3: View Test Examples (Optional)

- Scroll to sections 12.1-12.3 to see example assessments
- High-risk patient shows 7-10/10 score
- Low-risk patient shows 1-3/10 score
- Demonstrates realistic outcomes

---

## Common Questions

**Q: Is this a diagnosis?**
A: No. This is a screening tool to identify diabetes risk. Formal diagnosis requires clinical evaluation and glucose testing.

**Q: Can the score change?**
A: Yes! Risk factors like weight, exercise, and health status can improve with interventions. Reassess periodically (every 3-6 months for high-risk patients).

**Q: What if a patient refuses to answer some questions?**
A: All questions must be answered for an accurate risk score. Work with patient to understand concerns or barriers.

**Q: How accurate is this tool?**
A: The model correctly classifies ~76% of patients. It's designed to be sensitive (catch risky patients) rather than perfectly specific.

**Q: Can I use this for population screening?**
A: Yes! The functions support batch processing multiple patients. See example workflow in section 14.

**Q: Is patient data secure?**
A: The system only processes responses needed for calculation. Ensure proper security if storing results in EHR.

---

## Troubleshooting

### Issue: "Model not found" error

**Solution**: Verify the trained model file exists at `models/xgb_diabetes_model.json`

### Issue: "Field not in VALUE_TEXT_MAP" error

**Solution**: Check that all questionnaire field names match configuration in `config/VALUE_TEXT_MAP.json`

### Issue: Widgets not appearing

**Solution**: Ensure `ipywidgets` installed: `pip install ipywidgets`

### Issue: Risk score seems incorrect

**Solution**: 
- Verify all responses are selected
- Check that response labels exactly match configuration
- Test with example patients (section 12) to validate

---

## Next Steps

After generating assessment results:

1. **For Low Risk**: Continue preventive care, recheck annually
2. **For Moderate Risk**: Lifestyle counseling (diet/exercise), recheck in 6 months
3. **For High Risk**: Diabetes prevention program referral, glucose testing, intensive monitoring
4. **For Very High Risk**: Urgent referral, consider medication, endocrinology consultation

---

## Support & Feedback

For issues or feature requests:

- Consult the Function Reference (Section 14)
- Review Clinical Implementation Guide (Section 13)
- Check example workflows and interpretations (Sections 12-14)
- Validate with test cases included in this notebook

---
